# SOLiD-PINN v0.8.6 — Dynamic current and neural ODE reproduction
Select CPU, Run all, mount Drive. No PAT, no private data, no GitHub writes. All 60 fits and 4 targeted tests must pass before saving the synthetic results. Download the JSON for ChatGPT review. Includes negative findings. This is neural ODE, not a dynamic PINN or experimental validation.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess,sys,os,tempfile,json,hashlib
drive.mount("/content/drive")
REV="6879d8b4b9bec8ffac6ad800e911632a1788ae4e"
REPO=Path(tempfile.mkdtemp(prefix="solid-pinn-v086-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
source=(REPO/"scripts/dynamic_benchmark.py").read_bytes()
tests=(REPO/"scripts/test_dynamic_benchmark.py").read_bytes()
assert hashlib.sha256(source).hexdigest()=="526d3ab253e77e1adf8cbc3b1dc1a56b8317e5e0d9b4210814fa9479a799fc73"
assert hashlib.sha256(tests).hexdigest()=="ba6c280cdc86952410267a97a4a3c471c0255b453262e85bc8f2d5585283448e"
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_dynamic_benchmark.py"],cwd=REPO,env=env,check=True)
out=REPO/"results.json"
subprocess.run([sys.executable,"scripts/dynamic_benchmark.py","--out",str(out)],cwd=REPO,env=env,check=True)
d=json.loads(out.read_text())
expected={(t,n,s,m) for t in ("rational","exponential") for n in (1,5) for s in (11,23,37) for m in ("stateless","misspecified","pchip","oracle","neural_ode")}
assert len(d["metrics"])==60 and {(x["truth"],x["noise_mv"],x["seed"],x["method"]) for x in d["metrics"]}==expected
assert all(x["success"] and 0<=x["min_state"]<=x["max_state"]<=1 for x in d["metrics"])
assert len(d["checks"])==2 and all(x["rk4_state_rmse"]<1e-6 for x in d["checks"])
d["source_revision"]=REV
d["source_sha256"]=hashlib.sha256(source).hexdigest()
d["test_sha256"]=hashlib.sha256(tests).hexdigest()
d["targeted_tests_passed"]=4
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v086/handoff")
root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v086_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    part=dest.with_suffix(".part");part.write_bytes(raw);part.replace(dest)
assert dest.read_bytes()==raw
print("DRIVE VERIFIED",dest.name,sha)
files.download(str(dest))